In [1]:
import intake
import xarray as xr
import os

In [2]:
def load_cmip6_data(
    model,
    variable,
    data_source='cmip6',
    experiment='ssp370',
    temporal_resolution='day',
    variant='r1i1p1f2'  # note: this may vary depending on the model
):
    # Load the Pangeo CMIP6 catalog
    print("Loading the Pangeo CMIP6 catalog...")
    col_url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
    col = intake.open_esm_datastore(col_url)

    # Filter the dataset based on input parameters
    print("Filtering the dataset...")
    query = col.search(
        source_id=model,
        variable_id=variable,
        table_id=temporal_resolution,
        experiment_id=experiment,
        member_id=variant,
        grid_label="gr",  # you can change this to 'gr' if needed
    )

    if query.df.empty:
        print("⚠️ No dataset found with these parameters.")
        return None

    print("Loading data as an xarray dataset...")
    # Load the dataset(s) (this can take time and memory)
    dset_dict = query.to_dataset_dict(zarr_kwargs={'consolidated': True})
    
    # Usually, there's only one dataset per specific combination
    ds = list(dset_dict.values())[0]
    
    print("✅ Dataset successfully loaded.")
    return ds

In [3]:
exp = 'ssp370' 
model = 'CNRM-ESM2-1'
output_path = f'../data/cds_downloads/{exp}/{model}/'
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

In [5]:
# Example usage
if __name__ == "__main__":
    ds = load_cmip6_data(
        model='CNRM-ESM2-1',
        variable='vas',#uas
        experiment='ssp585',
        temporal_resolution='day',
        variant='r1i1p1f2' 
    )

    if ds:
        #print(ds)
        ds.to_netcdf(f'{output_path}')

Loading the Pangeo CMIP6 catalog...
Filtering the dataset...
Loading data as an xarray dataset...

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


✅ Dataset successfully loaded.


PermissionError: [Errno 13] Permission denied: '/home/python/jupyters/WMO_ATLAS/Notebook_for_Deliverable_2/data/cds_downloads/ssp370/CNRM-ESM2-1'